# P3 & P6 + CE-selector gold-free · MIMIC 3/6/10% + IU 3% (+ ablation P3)

Chạy P3/P6 (hoặc ablation của P3) với **selector 4-cách gold-free S1–S4**. Mỗi run tự
snapshot ứng viên LoRA + eval trên D_t_final. **Cell 2**: chọn DATASET/FORGET_PCT; bật
`RUN_ABLATION` để chạy ablation P3 thay vì P3+P6 core.


In [ ]:
# Cell 1: setup + CHOT CHAN code da push
import os, subprocess
WORK='/kaggle/working'; REPO=f'{WORK}/Forget-MI-LoKU'
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','https://github.com/nhnhu146/Forget-MI-LoKU.git',REPO],check=True)
else:
    subprocess.run(['git','-C',REPO,'pull','--ff-only'],check=True)
os.chdir(REPO)
assert os.path.exists('training/adv_common.py'),'push code truoc + re-import notebook'
_adv=open('training/adv_common.py').read()
assert 'ce_selector' in _adv and 'checkpoint_selection_' in _adv, \
    '❌ adv_common CHUA co hook CE-selector -> chay `git push` code MOI roi moi Save Version!'
assert 'OnlineCESelector' in open('training/ce_selector_pilot.py').read(), '❌ git push code moi truoc!'
print('✅ Code CE-selector da co (hook + OnlineCESelector).')
subprocess.run(['pip','install','-q','pydicom','scikit-image','scikit-learn','pyyaml','wandb','seaborn==0.13.2'],check=True)
subprocess.run(['pip','install','-q','transformers==4.38.0','peft==0.10.0','accelerate==0.27.0'],check=True)
import torch; assert torch.cuda.is_available(),'Bat GPU'
print('Commit:',subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip())
print('GPU   :',torch.cuda.get_device_name(0))


In [ ]:
# Cell 2: CHON + path discovery (mimic/iu) + ablation
import glob, os
DATASET    = 'mimic'   # 'mimic' | 'iu'
FORGET_PCT = 3         # 3 | 6 | 10  (iu: 3)
SEED       = 42

# --- ABLATION (P3 la phuong phap de xuat) ---
RUN_ABLATION = False   # False: chay P3+P6 core. True: chay P3-full + cac ablation duoi (MIMIC 3%).
ABLATIONS    = ['no_fila','no_uumu','no_noise']   # no_fila (bo Fisher/FILA init) | no_uumu | no_noise

assert DATASET in ('mimic','iu') and FORGET_PCT in (3,6,10)
if DATASET=='iu': assert FORGET_PCT==3,'IU chi co 3%'

def fd(*slugs):
    for s in slugs:
        if os.path.isdir(f'/kaggle/input/{s}'): return f'/kaggle/input/{s}'
        h=glob.glob(f'/kaggle/input/datasets/*/{s}')
        if h: return sorted(h)[0]
    return None
def bins(root): return sorted(glob.glob(os.path.join(root,'**','pytorch_model.bin'),recursive=True),key=len)

tag=f'{DATASET}{FORGET_PCT}per'
OUT=f'/kaggle/working/adv_ce_{tag}_s{SEED}'          # moi run mot thu muc con -> khong de nhau
RESULTS=f'/kaggle/working/results_adv_ce_{tag}.csv'

if DATASET=='mimic':
    CONFIG='config_advanced_kaggle.yaml'
    DATA=fd('forget-mi-data'); MOD=fd('forget-mi-models-full','forget-mi-models')
    assert DATA and MOD,'Add forget-mi-data + forget-mi-models-full'
    BASE=os.path.dirname([b for b in bins(MOD) if 'training_original_model' in b][0])
    gh=[b for b in bins(MOD) if f'model_retrained_{FORGET_PCT}per' in b]
    GOLD=os.path.dirname(gh[0]) if gh else BASE; HAS_GOLD=bool(gh)
    TEXT=os.path.join(DATA,'data','metadata'); IMG=os.path.join(DATA,'data','img_data')
    SPLIT='./data_splits/mimic-cxr-sub-img-edema-split-manualtest.csv'
    FORGET=f'./data_splits/forget_set_{FORGET_PCT}per.csv'
else:
    CONFIG='config_loku_iu_kaggle.yaml'
    DATA=fd('forget-mi-data-iu'); MOD=fd('forget-mi-models-iu'); MODRE=fd('forget-mi-models-iu-re')
    RAD=fd('chest-xrays-indiana-university')
    assert DATA and MOD and RAD,'Add forget-mi-data-iu + forget-mi-models-iu + forget-mi-models-iu-re + chest-xrays-indiana-university'
    ogb=[b for b in bins(MOD) if 'model_og' in b.lower() or 'base_model' in b.lower()] or bins(MOD)
    BASE=os.path.dirname(ogb[0])
    reb=(bins(MODRE) if MODRE else []) or [b for b in bins(MOD) if 'retrain' in b.lower()]
    GOLD=os.path.dirname(reb[0]) if reb else BASE; HAS_GOLD=bool(reb)
    def first_existing(root, rels):
        for r in rels:
            p=os.path.join(root,r)
            if os.path.exists(p): return p
        return None
    tsv=glob.glob(os.path.join(DATA,'**','all_data.tsv'),recursive=True) or glob.glob('/kaggle/input/**/all_data.tsv',recursive=True)
    TEXT=os.path.dirname(tsv[0]) if tsv else first_existing(DATA,['data/metadata','metadata'])
    IMG=first_existing(DATA,['data/img_data','img_data']) or (first_existing(RAD,['images/images_normalized','images']) if RAD else None) or RAD
    sp=glob.glob(os.path.join(DATA,'**','iu-split.csv'),recursive=True) or glob.glob('/kaggle/input/**/iu-split.csv',recursive=True) or glob.glob(os.path.join(DATA,'**','*iu*split*.csv'),recursive=True)
    fg=glob.glob(os.path.join(DATA,'**',f'forget_set_{FORGET_PCT}per_iu.csv'),recursive=True) or glob.glob(f'/kaggle/input/**/forget_set_{FORGET_PCT}per_iu.csv',recursive=True)
    assert sp and fg,f'Khong thay iu-split / forget_set_iu (glob toan input)'
    SPLIT=sp[0]; FORGET=fg[0]
    if not TEXT or not IMG:
        print('⚠️ TEXT',TEXT,'IMG',IMG,'- liet ke input:')
        for r,d,f in os.walk(DATA):
            if r[len(DATA):].count(os.sep)<=2: print(' ',r,'->',[x for x in f][:4])

for n,p in {'BASE':BASE,'TEXT':TEXT,'IMG':IMG,'SPLIT':SPLIT,'FORGET':FORGET}.items():
    assert p and os.path.exists(p),f'Missing {n}: {p}'
COMMON={'forget_set_path':FORGET,'base_model_path':BASE,'bert_pretrained_dir':BASE,
        'retrained_model_path':GOLD,'text_data_dir':TEXT,'img_data_dir':IMG,
        'data_split_path':SPLIT,'results_csv_path':RESULTS,'ce_selector':1,'s4_delta':0.15}
ABL_OVR={'no_fila':{'loku_random_init':1},'no_uumu':{'ablate_uu_mu':1},'no_noise':{'use_noise':0}}
print('DATASET',DATASET,'PCT',FORGET_PCT,'| config',CONFIG,'| tag',tag,'| GOLD',HAS_GOLD,'| ablation',RUN_ABLATION)
print('BASE',BASE); print('SPLIT',SPLIT); print('FORGET',FORGET)


In [ ]:
# Cell 3: chay JOBS (P3+P6 core, HOAC P3 ablations). Moi run 1 output_dir rieng.
import os, subprocess, time
SCRIPT={'p3':'training/forgetmi_p3.py','p6':'training/forgetmi_p6.py'}
if RUN_ABLATION:
    JOBS=[('p3',None)]+[('p3',a) for a in ABLATIONS]   # P3-full + ablation -> so truc tiep 1 luot
else:
    JOBS=[('p3',None),('p6',None)]              # core
LOG=[]
def run(m, abl):
    rid=f'{m}_{tag}_s{SEED}'+(f'_{abl}' if abl else '')
    od=f'{OUT}/{rid}'
    ovr=dict(COMMON); ovr['id']=rid; ovr['output_dir']=od
    ovr['history_csv_path']=f'/kaggle/working/perepoch_{rid}.csv'
    if abl: ovr.update(ABL_OVR[abl])
    arg=','.join(f'{k}={v}' for k,v in ovr.items())
    env={**os.environ,'PYTHONPATH':'.','WANDB_MODE':'disabled','PYTORCH_CUDA_ALLOC_CONF':'expandable_segments:True'}
    cmd=['python',SCRIPT[m],'--config',CONFIG,'--seed',str(SEED),'--fresh','--override',arg]
    print('='*70+f'\n{rid}\n'+'='*70); t0=time.time()
    try: subprocess.run(cmd,env=env,check=True); LOG.append((rid,'OK',round((time.time()-t0)/3600,2)))
    except subprocess.CalledProcessError as e: print('FAIL',rid,e.returncode); LOG.append((rid,f'FAIL{e.returncode}',0))
for m,a in JOBS: run(m,a)
print('\nTONG KET:'); [print(' ',*x) for x in LOG]


In [ ]:
# Cell 4: eval OG + GOLD tren D_t_final (chi khi CORE; ablation khong can lai)
import os, subprocess
def evalref(label, mpath):
    ovr=dict(COMMON); ovr.pop('ce_selector',None); ovr.pop('s4_delta',None)
    ovr['output_dir']=f'{OUT}/_ref'; ovr['results_csv_path']=RESULTS
    arg=','.join(f'{k}={v}' for k,v in ovr.items())
    env={**os.environ,'PYTHONPATH':'.','WANDB_MODE':'disabled'}
    cmd=['python','training/forgetmi_eval_only.py','--config',CONFIG,'--seed',str(SEED),
         '--label',label,'--model_type','pretrained','--model_path',mpath,'--method','reference','--override',arg]
    print('eval-ref',label)
    try: subprocess.run(cmd,env=env,check=True)
    except subprocess.CalledProcessError as e: print('FAIL',label,e.returncode)
if not RUN_ABLATION:
    evalref(f'og_{tag}',BASE)
    if HAS_GOLD: evalref(f're_{tag}',GOLD)
    else: print('(khong co GOLD cho',tag,')')
else:
    print('RUN_ABLATION -> bo qua eval-ref (dung OG/GOLD tu run core)')


In [ ]:
# Cell 5: tong hop (glob tat ca checkpoint_selection duoi OUT) + OG/GOLD
import os, json, glob, pandas as pd
pd.set_option('display.width',200)
rows=[]
for sj in sorted(glob.glob(f'{OUT}/**/checkpoint_selection_*/selected_checkpoints.json',recursive=True)):
    run_name=os.path.basename(os.path.dirname(os.path.dirname(sj)))   # ten thu muc run (id)
    res=json.load(open(sj))['results']
    for sel,v in res.items():
        if v.get('epoch') is None or 'Df_AUC' not in v: continue
        rows.append({'run':run_name,'selector':sel.replace('_',' '),'epoch':f"E{v['epoch']}",
                     'Df_AUC':v['Df_AUC'],'Df_F1':v['Df_F1'],'Dt_AUC':v['Dt_AUC'],'Dt_F1':v['Dt_F1'],'MIA':v['MIA']})
if os.path.exists(RESULTS):
    dr=pd.read_csv(RESULTS)
    for _,r in dr[dr.get('method')=='reference'].iterrows():
        rows.append({'run':str(r.get('run_id','ref')).split('_')[0].upper(),'selector':'(reference)','epoch':'-',
                     'Df_AUC':r.get('Forget_AUC'),'Df_F1':r.get('Forget_Macro_F1'),
                     'Dt_AUC':r.get('Test_AUC'),'Dt_F1':r.get('Test_Macro_F1'),'MIA':r.get('MIA')})
if rows:
    df=pd.DataFrame(rows); df.to_csv(f'/kaggle/working/summary_adv_ce_{tag}.csv',index=False)
    print(f'===== {tag} ====='); print(df.to_string(index=False))
print('\nTAI VE: summary_adv_ce_*.csv + results_adv_ce_*.csv + OUT/*/checkpoint_selection_*/*')
